# Prepare ISSP 2020 data for fine-tuning Kev

Loads `data/issp2020/issp2020.jsonl` and builds training records **on the fly** from structured fields, so the presentation can be augmented each time a record is used.

**Kev already does** (inside its trainer, every epoch, `kev/kev/data.py::augment`): shuffle option order of `choice` questions. **We add:** shuffle persona attribute order; reverse `score` (Likert) scales with the target remapped.

In [9]:
import json, random

records = [json.loads(line) for line in open("data/issp2020/issp2020.jsonl", encoding="utf-8")]
len(records)

13981

In [10]:
records[0]

{'group': 'AT/Female/15-29/Lower secondary or less',
 'country_code': 'AT',
 'block': 'Q1',
 'persona': {'Country': 'Austria',
  'Sex': 'Female',
  'Age': '15-29',
  'Education': 'Lower secondary or less'},
 'n': {'v1': 10},
 'questions': {'v1': {'type': 'choice',
   'criteria': {'Health care': None,
    'Education': None,
    'Crime': None,
    'The environment': None,
    'Immigration': None,
    'The economy': None,
    'Terrorism': None,
    'Poverty': None,
    'None of these': None},
   'instructions': 'Which of these issues is the most important for Austria today?',
   'label': 'Health care',
   'target': {'Health care': 0.4,
    'Education': 0.2,
    'The environment': 0.2,
    'Crime': 0.1,
    'None of these': 0.1},
   'src': 'issp2020'}}}

## Augmentation

- **Persona:** the state is rendered from the record's `persona` fields (the JSONL stores no state), with attributes in random order.
- **`score`:** with probability `p_reverse`, levels are reversed; target keys and label map *i* → *K−1−i*. At inference, scales must be shown low → high (Kev's `score` = expected level index).
- **`choice`:** left to Kev (it shuffles every epoch). **`noul`:** fixed (Kev always renders no, yes).

In [11]:
from issp_augment import augment, canonical   # the same code train_issp.py uses

## Check on one record: three views

In [12]:
r = next(r for r in records if r["block"] == "Q2")
rng = random.Random(0)
for i, p_rev in enumerate([0.0, 0.0, 1.0]):  # third view: scale reversed
    a = augment(r, rng, p_reverse=p_rev)
    q = a["questions"]["v4"] if "v4" in a["questions"] else next(iter(a["questions"].values()))
    print(f"--- view {i}\n{a['state']}")
    for k, level in enumerate(q["criteria"]):
        print(f"  {q['target'].get(str(k), 0):.3f}  {level}")
    print("  label:", q["label"], "=", q["criteria"][q["label"]])

--- view 0
Age: 15-29
Sex: Female
Country: Austria
Education: Lower secondary or less
  0.000  Disagree strongly
  0.000  Disagree
  0.000  Neither agree nor disagree
  0.545  Agree
  0.455  Agree strongly
  label: 3 = Agree
--- view 1
Country: Austria
Education: Lower secondary or less
Sex: Female
Age: 15-29
  0.000  Disagree strongly
  0.000  Disagree
  0.000  Neither agree nor disagree
  0.545  Agree
  0.455  Agree strongly
  label: 3 = Agree
--- view 2
Age: 15-29
Country: Austria
Education: Lower secondary or less
Sex: Female
  0.455  Agree strongly
  0.545  Agree
  0.000  Neither agree nor disagree
  0.000  Disagree
  0.000  Disagree strongly
  label: 1 = Agree
